# AHC015 random 128-channel student + 3-hour policy distillation

Save & Run All用Notebook。未来列なしafterstate 128 channelモデルを完全ランダム初期化し、学習済み64 channelモデルから未来補正を除いた盤面方策への `KL(teacher || student)` をPPO損失へ加える。蒸留係数は実時間3時間で `1 → 0` に線形減衰する。criticは蒸留せず、potential shapingされたPPO returnだけから学習する。studentにもteacherにも未来列は入力しない。

KaggleでGPU T4 x2、Internet Onを選び、`GITHUB_TOKEN` と `WANDB_API_KEY` のSecret accessを有効にして実行する。W&B run名は `small-<時刻>`。このNotebookは手順1専用で、手順2では生成された `best-training.pt` から蒸留なしで17時間resumeする。

In [ ]:
import torch

assert torch.cuda.is_available(), "CUDA GPU is required"
assert torch.cuda.device_count() == 2, "Select the Kaggle GPU T4 x2 accelerator"
for index in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(index)
    capability = torch.cuda.get_device_capability(index)
    print(index, name, capability)
    assert "T4" in name and capability == (7, 5), "GPU T4 x2 is required"

test_tensor = torch.zeros((8, 4, 10, 10), device="cuda")
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA tensor:", test_tensor.shape, test_tensor.device)
del test_tensor
torch.cuda.empty_cache()

In [ ]:
import base64
import os
import subprocess
from pathlib import Path

from kaggle_secrets import UserSecretsClient

repo_dir = Path("/kaggle/working/ahc-ml")
assert not repo_dir.exists(), f"Clean session required: {repo_dir} already exists"
github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
credentials = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraHeader"
git_env["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {credentials}"
try:
    subprocess.run(
        [
            "git", "clone", "--branch", "feature/ahc015-teacher",
            "--single-branch", "https://github.com/e1jirou/ahc-ml.git", str(repo_dir),
        ],
        check=True, env=git_env,
    )
finally:
    del github_token, credentials, git_env

actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True
).strip()
os.chdir(repo_dir)
print("Repository commit:", actual_commit)
print("Current directory:", os.getcwd())

In [ ]:
%pip install --quiet torchview==0.2.7

from importlib.metadata import version
print("torchview:", version("torchview"))

In [ ]:
import os
from pathlib import Path

import torch
import wandb
from kaggle_secrets import UserSecretsClient

wandb_api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
os.environ["WANDB_API_KEY"] = wandb_api_key
del wandb_api_key
assert wandb.login(verify=True)

api = wandb.Api()
source_run = api.run("eijirou-personal/ahc-ml/8vwhuzf0")
assert source_run.name == "small-20260902-102642"
assert source_run.state == "finished"
artifact_name = (
    "eijirou-personal/ahc-ml/"
    "small-20260902-102642-training-checkpoint:v0"
)
checkpoint_dir = Path("/kaggle/working/checkpoints") / source_run.name
artifact = api.artifact(artifact_name, type="model")
downloaded_dir = Path(artifact.download(root=checkpoint_dir))
teacher_checkpoint_path = downloaded_dir / "best-training.pt"
assert teacher_checkpoint_path.is_file()
teacher_checkpoint = torch.load(teacher_checkpoint_path, map_location="cpu", weights_only=False)
assert teacher_checkpoint["epoch"] == 457
assert teacher_checkpoint["config"]["model"]["channels"] == 64
assert teacher_checkpoint["config"]["model"]["future_mode"] == "full_late"
print("Teacher W&B run:", source_run.name, source_run.id)
print("Teacher checkpoint:", teacher_checkpoint_path)
print("Source model fixed mean score:", teacher_checkpoint["metrics"]["evaluation/mean_score"])
print("Board-only teacher ablation mean score: 787450.627")
del teacher_checkpoint

In [ ]:
# Fast preflight: verify the exact experiment before starting the 3-hour cell.
import sys

sys.path.insert(0, str(repo_dir / "python"))
from examples.ahc015.python.afterstate_model import AfterstatePpoNet
from examples.ahc015.python.config import load_config
from examples.ahc015.python.train import load_distillation_teacher

config_path = repo_dir / "examples/ahc015/config_afterstate_128_distill.toml"
config = load_config(config_path)
assert config.model.input_mode == "afterstate"
assert config.model.future_mode == "none"
assert (config.model.channels, config.model.residual_blocks) == (128, 10)
assert config.training.max_hours == 3.0
assert config.training.learning_rate == 1e-4
assert config.ppo.reward_mode == "potential_shaping"
assert config.ppo.policy_phi_coefficient_start == 0.0
assert config.distillation.coefficient_start == 1.0
assert config.distillation.coefficient_end == 0.0
assert config.distillation.anneal_hours == 3.0
student = AfterstatePpoNet(128, 10, "none")
teacher = load_distillation_teacher(teacher_checkpoint_path)
assert teacher.channels == 64 and teacher.residual_blocks == 10
assert torch.count_nonzero(student.actor.output.weight) == 0
assert torch.count_nonzero(student.actor.board_stem.weight) > 0
assert teacher.future_mode == "none"
print("Preflight OK: random 128ch student, extracted board-only 64ch teacher, no future input")
del student, teacher

In [ ]:
# Long-running cell: approximately 3 hours, then checkpoint upload to W&B.
import os
import subprocess
import sys

train_env = os.environ.copy()
train_env["PYTHONPATH"] = str(repo_dir / "python")
train_env["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
subprocess.run(
    [
        sys.executable, "-m", "examples.ahc015.python.train",
        "--config", "examples/ahc015/config_afterstate_128_distill.toml",
        "--distill-from", str(teacher_checkpoint_path),
    ],
    cwd=repo_dir, env=train_env, check=True,
)

In [ ]:
# Confirm outputs needed by the 17-hour continuation stage.
import json

run_dirs = sorted((repo_dir / "outputs/ahc015").glob("small-*"))
assert run_dirs
latest_run = run_dirs[-1]
best_training_path = latest_run / "best-training.pt"
last_training_path = latest_run / "last.pt"
assert best_training_path.is_file() and last_training_path.is_file()
with (latest_run / "metrics.jsonl").open() as file:
    final_metrics = json.loads(list(file)[-1])
saved = torch.load(best_training_path, map_location="cpu", weights_only=False)
assert saved["config"]["model"]["future_mode"] == "none"
assert saved["config"]["model"]["channels"] == 128
print("Completed run:", latest_run.name)
print("Elapsed hours:", final_metrics["timing/elapsed_hours"])
print("Final distillation coefficient:", final_metrics["training/distillation_coefficient"])
print("Final mean score:", final_metrics.get("evaluation/mean_score"))
print("Stage-2 checkpoint:", best_training_path)